In [ ]:
!pip install ultralytics -q
print("Ultralytics is installed")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.6 MB/s eta 0:00:00

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.4 MB/s eta 0:00:00a 0:00:01

Ultralytics is installed.

In [ ]:
import torch
assert torch.cuda.is_available()
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

GPU: Tesla T4

VRAM: 15.6 GB

In [ ]:
# Params for the first run
from pathlib import Path

DATASET_DIR  = "/kaggle/input/datasets/natair/chicken-dataset-v2/chicken_dataset_v2"
WORK_DIR     = "/kaggle/working"
RUN_NAME     = "chicken_detector_v2"
IMG_SIZE     = 1280
TOTAL_EPOCHS = 150
BATCH        = 8
MODEL_BASE   = "yolov8m.pt"

In [ ]:
# Params for fine-tuning
from ultralytics import YOLO
import os
from pathlib import Path

DATASET_DIR         = "/kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3"
WORK_DIR            = "/kaggle/working"
RUN_NAME            = "chicken_detector_v3"
IMG_SIZE            = 1280
TOTAL_EPOCHS        = 60
BATCH               = 8
FINETUNE_BASE_MODEL = "/kaggle/input/models/natair/chicken-label-drone-model-tolo8m-825frames/pytorch/default/1/best_l.pt"

Creating new Ultralytics Settings v0.0.6 file

View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'

Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

In [ ]:
# Creating a dataset config for YOLO (first run)
data_yaml = f"""
path: {DATASET_DIR}
train: images/train
val: images/val

names:
  0: chicken
"""

yaml_path = f"{WORK_DIR}/data.yaml"
with open(yaml_path, "w") as f:
    f.write(data_yaml)

print(f"data.yaml created: {yaml_path}")
print(data_yaml)

train_imgs = list(Path(DATASET_DIR, "images/train").glob("*.jpg"))
val_imgs   = list(Path(DATASET_DIR, "images/val").glob("*.jpg"))
train_lbls = list(Path(DATASET_DIR, "labels/train").glob("*.txt"))
val_lbls   = list(Path(DATASET_DIR, "labels/val").glob("*.txt"))

print(f"Train: {len(train_imgs)} images / {len(train_lbls)} annotaions")
print(f"Val:   {len(val_imgs)} images / {len(val_lbls)} annotaions")

assert len(train_imgs) > 0, "No train images found."
assert len(train_imgs) == len(train_lbls), "The number of images and annotations does not match!"


data.yaml created: /kaggle/working/data.yaml

path: /kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3
train: images/train
val: images/val
 
names:
  0: chicken

Train: 719 images / 719 annotaions
Val:   126 images / 126 annotaions

In [ ]:
# Creating a dataset config for YOLO (fine-tuning)
yaml_path      = "/kaggle/working/chicken_dataset_v3_boosted/data.yaml"
DATASET_DIR_v2 = "/kaggle/working/chicken_dataset_v3_boosted"

train_imgs = list(Path(DATASET_DIR_v2, "images/train").glob("*.jpg"))
val_imgs   = list(Path(DATASET_DIR_v2, "images/val").glob("*.jpg"))
train_lbls = list(Path(DATASET_DIR_v2, "labels/train").glob("*.txt"))
val_lbls   = list(Path(DATASET_DIR_v2, "labels/val").glob("*.txt"))

print(f"Train: {len(train_imgs)} images / {len(train_lbls)} annotaions")
print(f"Val:   {len(val_imgs)} images / {len(val_lbls)} annotaions")

assert len(train_imgs) > 0, "No train images found."
assert len(train_imgs) == len(train_lbls), "The number of images and annotations does not match!"


Train: 1910 images / 1910 annotaions

Val:   126 images / 126 annotaions

In [ ]:
# First training run
from ultralytics import YOLO
import os

RESUME_CHECKPOINT = None
# "/kaggle/input/chicken-checkpoint/last.pt"

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).exists():
    print(f"Continuing training from the checkpoint: {RESUME_CHECKPOINT}")
    model = YOLO(RESUME_CHECKPOINT)
    resume_flag = True
else:
    print(f"Start training from scratch: {MODEL_BASE}")
    model = YOLO(MODEL_BASE)
    resume_flag = False

results = model.train(
    data=yaml_path,
    epochs=TOTAL_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name=RUN_NAME,
    project=f"{WORK_DIR}/runs",
    resume=resume_flag,

    # Augmentations for small objects
    mosaic=1.0,
    scale=0.9,
    copy_paste=0.3,
    degrees=10.0,
    fliplr=0.5,
    flipud=0.3,
    hsv_h=0.015,
    hsv_v=0.4,

    # Session interruption resilience
    save=True,
    save_period=5,      # checkpoint every 5 epochs
    patience=40,        # early discontinuation if there is no improvement over a prolonged period
    exist_ok=True,      # write to the same `run` folder again
    verbose=True,
    plots=True,
)

print("\n Training completed (or interrupted due to patience)")
print(f"  best.pt: {WORK_DIR}/runs/{RUN_NAME}/weights/best.pt")
print(f"  last.pt: {WORK_DIR}/runs/{RUN_NAME}/weights/last.pt")

Creating new Ultralytics Settings v0.0.6 file 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Start training from scratch: yolov8m.pt
Downloading https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8m.pt to 'yolov8m.pt': 100% ━━━━━━━━━━━━ 49.7MB 324.4MB/s 0.2s0.1s<0.1s
Ultralytics 8.4.88  Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=chicken_detector_v2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=40, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=/kaggle/working/runs, quantize=None, rect=False, resume=False, retina_masks=False, rle=1.0, save=True, save_conf=False, save_crop=False, save_dir=/kaggle/working/runs/chicken_detector_v2, save_frames=False, save_json=False, save_period=5, save_txt=False, scale=0.9, seed=0, shear=0.0, show=False, show_boxes=True, show_conf=True, show_labels=True, simplify=True, single_cls=False, source=None, split=val, stream_buffer=False, task=detect, time=None, tracker=tracktrack.yaml, translate=0.1, val=True, verbose=True, vid_stride=1, visualize=False, warmup_bias_lr=0.1, warmup_epochs=3.0, warmup_momentum=0.8, weight_decay=0.0005, workers=8, workspace=None
Downloading https://ultralytics.com/assets/Arial.ttf to '/root/.config/Ultralytics/Arial.ttf': 100% ━━━━━━━━━━━━ 755.1KB 133.8MB/s 0.0s
Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics.nn.modules.conv.Conv             [384, 576, 3, 2]              
  8                  -1  2   3985920  ultralytics.nn.modules.block.C2f             [576, 576, 2, True]           
  9                  -1  1    831168  ultralytics.nn.modules.block.SPPF            [576, 576, 5]                 
 10                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 11             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 12                  -1  2   1993728  ultralytics.nn.modules.block.C2f             [960, 384, 2]                 
 13                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 14             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 15                  -1  2    517632  ultralytics.nn.modules.block.C2f             [576, 192, 2]                 
 16                  -1  1    332160  ultralytics.nn.modules.conv.Conv             [192, 192, 3, 2]              
 17            [-1, 12]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 18                  -1  2   1846272  ultralytics.nn.modules.block.C2f             [576, 384, 2]                 
 19                  -1  1   1327872  ultralytics.nn.modules.conv.Conv             [384, 384, 3, 2]              
 20             [-1, 9]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 21                  -1  2   4207104  ultralytics.nn.modules.block.C2f             [960, 576, 2]                 
 22        [15, 18, 21]  1   3776275  ultralytics.nn.modules.head.Detect           [1, 16, None, [192, 384, 576]]
Model summary: 170 layers, 25,856,899 parameters, 25,856,883 gradients, 79.1 GFLOPs

Transferred 469/475 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
Downloading https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt to 'yolo26n.pt': 100% ━━━━━━━━━━━━ 5.3MB 268.2MB/s 0.0s
AMP: checks passed 
WARNING train: Slow image access detected (ping: 0.0±0.0 ms, read: 30.4±5.7 MB/s, size: 132.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
train: Scanning /kaggle/input/datasets/natair/chicken-dataset-v2/chicken_dataset_v2/labels/train... 719 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 719/719 287.7it/s 2.5s0.0s
WARNING train: Cache directory /kaggle/input/datasets/natair/chicken-dataset-v2/chicken_dataset_v2/labels is not writable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
WARNING val: Slow image access detected (ping: 0.3±0.4 ms, read: 32.6±4.0 MB/s, size: 138.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/input/datasets/natair/chicken-dataset-v2/chicken_dataset_v2/labels/val... 126 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 126/126 268.9it/s 0.5s0.1s
WARNING val: Cache directory /kaggle/input/datasets/natair/chicken-dataset-v2/chicken_dataset_v2/labels is not writable, cache not saved.
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Plotting labels to /kaggle/working/runs/chicken_detector_v2/labels.jpg... 
Image sizes 1280 train, 1280 val
Using 2 dataloader workers
Logging results to /kaggle/working/runs/chicken_detector_v2
Starting training for 150 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/150        13G      1.972      2.121      1.274         78       1280: 100% ━━━━━━━━━━━━ 90/90 1.0s/it 1:301.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.8it/s 4.5s0.5ss
                   all        126       1080      0.106      0.756     0.0938     0.0463

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/150      12.3G      1.663      1.345      1.148         77       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:391.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.9it/s 4.1s0.6ss
                   all        126       1080      0.546      0.431      0.409      0.172

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/150      12.4G      1.709      1.283      1.128         85       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.0it/s 4.1s0.6ss
                   all        126       1080      0.636       0.47       0.54      0.266

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/150      11.6G      1.708      1.276      1.134        111       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.768      0.648      0.717      0.429

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      5/150        12G       1.58      1.117      1.099         65       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.742      0.678      0.733      0.389

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      6/150      11.9G      1.516      1.061      1.077         67       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.711      0.581      0.669      0.398

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      7/150      12.3G      1.559      1.069      1.087         79       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.678      0.631      0.698      0.415

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      8/150      12.2G      1.508      1.009      1.061         40       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.827      0.757      0.842      0.475

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      9/150      12.1G      1.484     0.9623      1.072         82       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.767      0.688      0.769      0.416

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     10/150      11.9G       1.52     0.9625      1.056        139       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.791      0.717      0.787      0.419

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     11/150      11.9G      1.456     0.9724      1.042        139       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.817      0.723      0.818      0.498

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     12/150      12.1G      1.474     0.9642      1.051         23       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.793      0.722       0.81      0.451

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     13/150      11.7G      1.433     0.8922      1.046        150       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.804      0.718      0.826      0.465

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     14/150        12G      1.428      0.909      1.046        174       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.803      0.753      0.823       0.47

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     15/150      12.3G      1.437     0.9013      1.037        148       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.822      0.775      0.845      0.501

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     16/150      12.1G       1.38     0.8635      1.019         70       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.807      0.769      0.839      0.486

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     17/150      12.3G      1.449     0.9058      1.036         79       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.761      0.694      0.792      0.484

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     18/150      12.3G      1.517     0.9337      1.065        117       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.761      0.718      0.802      0.505

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     19/150        12G      1.469     0.9094      1.048        111       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.839      0.763      0.848        0.5

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     20/150      12.3G      1.392     0.8622      1.029         87       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.848      0.746      0.844      0.507

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     21/150      11.8G      1.359     0.8328      1.008        109       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.823      0.743      0.831      0.499

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     22/150      11.8G      1.373     0.8663      1.033         54       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1080      0.826      0.765      0.842      0.529

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     23/150      11.7G      1.381     0.8319      1.008         42       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.829      0.778      0.863      0.539

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     24/150      12.3G      1.355     0.8299      1.012         95       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.866      0.736      0.859      0.544

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     25/150        12G      1.342     0.7985      1.004         89       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1080      0.826      0.802      0.875      0.492

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     26/150      12.4G      1.377     0.7971      1.009         89       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.849      0.769      0.854      0.548

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     27/150        12G      1.423     0.8631      1.027        105       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1080      0.803      0.747      0.827      0.487

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     28/150      12.2G      1.353     0.7966      1.014         83       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.843      0.771      0.866      0.543

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     29/150      12.3G      1.354     0.8233      1.001        129       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.879      0.761      0.875      0.546

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     30/150      11.8G       1.33     0.7918     0.9945        122       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.857      0.816      0.892      0.545

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     31/150      12.4G      1.306     0.7778     0.9965         96       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.844      0.781       0.88      0.565

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     32/150      11.8G      1.313     0.7646     0.9993        154       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.864      0.789      0.874       0.54

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     33/150      12.3G      1.322     0.8047      1.001         55       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.838      0.788      0.873      0.543

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     34/150      11.9G      1.315     0.7763     0.9871        164       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1080      0.821      0.785      0.863      0.518

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     35/150      11.6G      1.312     0.7728      1.005        193       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.808      0.805      0.867      0.535

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     36/150      12.1G      1.326     0.7739      0.997        134       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.841       0.79       0.87      0.522

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     37/150      12.1G      1.286     0.7782     0.9944         81       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.855      0.816      0.894      0.562

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     38/150        12G       1.31     0.7801     0.9991        127       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1080      0.844      0.809      0.888      0.555

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     39/150      11.9G      1.291     0.7735     0.9916        110       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.854      0.799      0.886      0.495

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     40/150      11.9G      1.312     0.7901     0.9888        130       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.835      0.782      0.874      0.552

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     41/150        12G      1.276     0.7645     0.9712        131       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.871      0.798      0.888      0.556

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     42/150      12.1G      1.299     0.7446     0.9875        131       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.868      0.794      0.885      0.537

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     43/150      11.9G      1.321     0.7784     0.9965         45       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.862      0.814      0.893      0.569

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     44/150      12.4G      1.269     0.7508     0.9796         84       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:381.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.847      0.786      0.872      0.556

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     45/150      12.1G      1.265     0.7527     0.9894        175       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.864      0.745      0.853      0.568

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     46/150      12.2G      1.269      0.738     0.9741         97       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.873      0.796      0.892      0.583

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     47/150      11.6G      1.247      0.721     0.9873        137       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.864      0.809      0.893      0.592

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     48/150      11.9G      1.282     0.7412     0.9818        125       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.869      0.793      0.883      0.541

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     49/150      12.2G       1.27     0.7366     0.9726         76       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080       0.84      0.782      0.863      0.546

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     50/150      12.3G      1.251     0.7264     0.9738         98       1280: 100% ━━━━━━━━━━━━ 90/90 1.1s/it 1:371.0sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1080      0.852      0.806      0.882      0.584

In [ ]:
# Fine-tuning run
RESUME_CHECKPOINT = None
# "/kaggle/input/chicken-checkpoint-v3/last.pt"

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).exists():
    print(f"Continuing interrupted fine-tuning from: {RESUME_CHECKPOINT}")
    model = YOLO(RESUME_CHECKPOINT)
    resume_flag = True
else:
    print(f"Starting fine-tuning from: {FINETUNE_BASE_MODEL}")
    model = YOLO(FINETUNE_BASE_MODEL)
    resume_flag = False

results = model.train(
    data=yaml_path,
    epochs=80,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name="chicken_detector_v3_boosted",
    project=f"{WORK_DIR}/runs",
    resume=resume_flag,
    lr0=0.005,
    lrf=0.01,
    mosaic=1.0,
    scale=0.9,
    copy_paste=0.3,
    degrees=10.0,
    fliplr=0.5,
    flipud=0.3,
    hsv_h=0.015,
    hsv_v=0.4,
    save=True,
    save_period=5,
    patience=50,
    exist_ok=True,
    verbose=True,
    plots=True,
)

print("\n Fine-tuning completed (or interrupted due to patience)")
print(f"  best.pt: {WORK_DIR}/runs/chicken_detector_v3_boosted/weights/best.pt")
print(f"  last.pt: {WORK_DIR}/runs/chicken_detector_v3_boosted/weights/last.pt")

Starting fine-tuning from: /kaggle/input/models/natair/chicken-label-drone-model-tolo8m-825frames/pytorch/default/1/best_l.pt
Ultralytics 8.4.95 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/chicken_dataset_v3_boosted/data.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/models/natair/chicken-label-drone-model-tolo8m-825frames/pytorch/default/1/best_l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=chicken_detector_v3_boosted, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=50, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=/kaggle/working/runs, quantize=None, rect=False, resume=False, retina_masks=False, rle=1.0, save=True, save_conf=False, save_crop=False, save_dir=/kaggle/working/runs/chicken_detector_v3_boosted, save_frames=False, save_json=False, save_period=5, save_txt=False, scale=0.9, seed=0, shear=0.0, show=False, show_boxes=True, show_conf=True, show_labels=True, simplify=True, single_cls=False, source=None, split=val, stream_buffer=False, task=detect, time=None, tracker=tracktrack.yaml, translate=0.1, val=True, verbose=True, vid_stride=1, visualize=False, warmup_bias_lr=0.1, warmup_epochs=3.0, warmup_momentum=0.8, weight_decay=0.0005, workers=8, workspace=None

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics.nn.modules.conv.Conv             [384, 576, 3, 2]              
  8                  -1  2   3985920  ultralytics.nn.modules.block.C2f             [576, 576, 2, True]           
  9                  -1  1    831168  ultralytics.nn.modules.block.SPPF            [576, 576, 5]                 
 10                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 11             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 12                  -1  2   1993728  ultralytics.nn.modules.block.C2f             [960, 384, 2]                 
 13                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 14             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 15                  -1  2    517632  ultralytics.nn.modules.block.C2f             [576, 192, 2]                 
 16                  -1  1    332160  ultralytics.nn.modules.conv.Conv             [192, 192, 3, 2]              
 17            [-1, 12]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 18                  -1  2   1846272  ultralytics.nn.modules.block.C2f             [576, 384, 2]                 
 19                  -1  1   1327872  ultralytics.nn.modules.conv.Conv             [384, 384, 3, 2]              
 20             [-1, 9]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 21                  -1  2   4207104  ultralytics.nn.modules.block.C2f             [960, 576, 2]                 
 22        [15, 18, 21]  1   3776275  ultralytics.nn.modules.head.Detect           [1, 16, None, [192, 384, 576]]
Model summary: 170 layers, 25,856,899 parameters, 25,856,883 gradients, 79.1 GFLOPs

Transferred 475/475 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed 
train: Fast image access (ping: 0.0±0.0 ms, read: 2587.1±896.3 MB/s, size: 140.8 KB)
train: Scanning /kaggle/working/chicken_dataset_v3_boosted/labels/train... 1910 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1910/1910 1.1Kit/s 1.8s0.1s
train: New cache created: /kaggle/working/chicken_dataset_v3_boosted/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.0±0.0 ms, read: 829.7±638.2 MB/s, size: 133.9 KB)
val: Scanning /kaggle/working/chicken_dataset_v3_boosted/labels/val... 126 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 126/126 1.2Kit/s 0.1s<0.0s
val: New cache created: /kaggle/working/chicken_dataset_v3_boosted/labels/val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.005' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Plotting labels to /kaggle/working/runs/chicken_detector_v3_boosted/labels.jpg... 
Image sizes 1280 train, 1280 val
Using 2 dataloader workers
Logging results to /kaggle/working/runs/chicken_detector_v3_boosted
Starting training for 80 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/80      14.1G      1.308     0.7266     0.9725        122       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:131.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.0it/s 3.9s0.5ss
                   all        126       1355      0.863      0.754      0.841      0.541

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/80      12.4G      1.367     0.7707     0.9869        105       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:181.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.841      0.733      0.831      0.523

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/80      12.6G      1.391     0.7983     0.9971        195       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:181.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.835      0.776      0.857      0.528

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/80      13.3G      1.425     0.8114      1.004        110       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355       0.85      0.768      0.852      0.529

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       5/80      13.8G      1.386     0.7787     0.9964         95       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.864      0.772      0.864      0.531

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       6/80        13G      1.398     0.7789     0.9956        116       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.828       0.77      0.851      0.538

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       7/80      13.2G      1.399     0.7812     0.9954        158       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.859      0.735      0.851      0.528

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       8/80      13.4G       1.39      0.775     0.9912         42       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.844      0.767      0.851      0.517

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       9/80      12.5G      1.394     0.7742     0.9879        113       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.846      0.782      0.852      0.521

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      10/80      12.4G       1.38     0.7523     0.9895         93       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.847      0.755       0.85      0.536

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      11/80      12.5G      1.365     0.7585     0.9838        112       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.847      0.768      0.854      0.532

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      12/80      12.7G      1.357     0.7507     0.9838        171       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.862      0.763      0.857      0.524

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      13/80      13.8G      1.368     0.7533     0.9816        171       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.848      0.785      0.862      0.545

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      14/80      12.7G       1.36       0.74     0.9823        165       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.839      0.773      0.853       0.53

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      15/80      12.6G      1.358     0.7348     0.9826         76       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.858       0.78      0.862      0.543

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      16/80      13.3G      1.331     0.7191      0.979         72       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.847      0.797      0.864      0.556

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      17/80      12.9G      1.354     0.7251     0.9867        189       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.867      0.771      0.861      0.542

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      18/80      12.8G      1.342     0.7307     0.9771        128       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.855      0.779      0.859      0.557

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      19/80      13.6G      1.327     0.7117     0.9709        119       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355       0.85        0.8      0.871      0.553

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      21/80      12.8G      1.327     0.7039     0.9714        149       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.854      0.769      0.859      0.552

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      22/80      13.2G      1.307     0.6931     0.9717         49       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.884       0.78      0.871      0.555

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      23/80      13.1G      1.337     0.7071     0.9803        201       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.843      0.813       0.87      0.547

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      24/80      13.3G      1.327     0.7096     0.9782        204       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.858      0.779      0.861      0.551

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      25/80      12.5G      1.308     0.6928      0.964         97       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.859      0.801      0.873      0.553

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      26/80      12.6G      1.306      0.684     0.9691        113       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.884      0.773      0.868      0.556

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      27/80      13.6G      1.317     0.6964     0.9623        132       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.863      0.804      0.872      0.541

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      28/80      13.1G        1.3     0.6776     0.9655         84       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.874      0.787      0.871      0.556

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      29/80      13.3G      1.281     0.6694     0.9583        209       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.849      0.804      0.875      0.559

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      30/80      12.7G      1.286     0.6672     0.9601        157       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.859      0.812       0.88      0.555

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      31/80      13.1G      1.282     0.6625     0.9573        123       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.869      0.793      0.875      0.561

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      32/80      12.5G      1.265     0.6542       0.96         76       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.868      0.786      0.868      0.553

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      33/80        13G      1.281     0.6668      0.959        133       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.847      0.821      0.879      0.566

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      34/80      12.5G      1.287     0.6632     0.9525         77       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.863      0.801      0.876       0.56

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      35/80      12.9G      1.274     0.6532     0.9576        100       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.843      0.801      0.866      0.547

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      36/80      13.6G      1.267     0.6472     0.9483         75       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.876      0.803       0.88      0.562

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      37/80      13.3G      1.263     0.6494     0.9502         38       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.845      0.797      0.871      0.561

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      38/80      12.9G      1.255     0.6409     0.9495         61       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.863      0.782      0.871      0.559

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      39/80      12.8G      1.257     0.6357     0.9471         64       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.869      0.801      0.873      0.559

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      40/80      12.3G      1.242     0.6265     0.9466        191       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.873      0.816      0.884      0.557

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      41/80      12.8G       1.23     0.6302      0.946        133       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:161.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.867      0.806      0.878      0.563

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      42/80      13.2G      1.233     0.6189     0.9381        166       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.885      0.809      0.884      0.566

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      43/80        13G      1.243     0.6258     0.9423        187       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.866      0.808      0.873      0.557

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      44/80      12.4G      1.223     0.6156     0.9331        240       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.873      0.815      0.884      0.565

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      45/80      13.2G      1.216      0.611     0.9439        126       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.879      0.798      0.876      0.562

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      46/80      12.8G      1.231     0.6126     0.9436        132       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.876      0.796      0.877      0.557

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      47/80      13.4G      1.211     0.6057     0.9376        113       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.862       0.81      0.882      0.563

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      48/80      13.4G      1.211      0.606     0.9312         55       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.9s0.5ss
                   all        126       1355      0.852      0.821      0.878      0.563

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      49/80      12.5G      1.215     0.6075     0.9342        123       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.854      0.819      0.876      0.561

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      50/80      12.5G      1.207     0.6015     0.9409        143       1280: 100% ━━━━━━━━━━━━ 239/239 1.1s/it 4:171.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s0.5ss
                   all        126       1355      0.869      0.818       0.88      0.563

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = f"{WORK_DIR}/runs/{RUN_NAME}_boosted/results.csv"
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print("Last 5 epochs:")
print(df[["epoch", "metrics/precision(B)", "metrics/recall(B)",
          "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
axes[0].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
axes[0].set_title("mAP by epochs")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df["epoch"], df["metrics/precision(B)"], label="Precision")
axes[1].plot(df["epoch"], df["metrics/recall(B)"], label="Recall")
axes[1].set_title("Precision / Recall")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{WORK_DIR}/training_summary.jpg", dpi=120)
plt.show()

best_map50 = df["metrics/mAP50(B)"].max()
print(f"\nBest mAP50: {best_map50:.3f}")

Last 5 epochs:

    epoch  metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  \

75     76               0.87218            0.81583           0.87625   
76     77               0.87434            0.81651           0.87985   
77     78               0.87896            0.81255           0.87784   
78     79               0.87843            0.81591           0.87800   
79     80               0.87658            0.81993           0.87962   

    metrics/mAP50-95(B)  
    
75              0.55637  
76              0.55881  
77              0.55947  
78              0.56088  
79              0.56238  

![](../../data/images/Screenshot%202026-09-15%20at%2000.21.59.png)

Best mAP50: 0.884

A good result for a first model.

In [ ]:
best_model = YOLO(f"{WORK_DIR}/runs/{RUN_NAME}_boosted/weights/best.pt")

# Picking any image from 'val' for a visual check.
test_img = str(val_imgs[0])
res = best_model(test_img, imgsz=IMG_SIZE, conf=0.25)

res[0].save(filename=f"{WORK_DIR}/test_prediction.jpg")
print(f"Result saved: {WORK_DIR}/test_prediction.jpg")
print(f"Objects found: {len(res[0].boxes)}")

import cv2
img = cv2.cvtColor(cv2.imread(f"{WORK_DIR}/test_prediction.jpg"), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 9))
plt.imshow(img)
plt.axis("off")
plt.title(f"Detections: {len(res[0].boxes)}")
plt.show()

image 1/1 /kaggle/working/chicken_dataset_v3_boosted/images/val/frame_0652.jpg: 736x1280 4 chickens, 72.7ms

Speed: 6.0ms preprocess, 72.7ms inference, 1.3ms postprocess per image at shape (1, 3, 736, 1280)

Result saved: /kaggle/working/test_prediction.jpg

Objects found: 4

![](../../data/images/Screenshot%202026-09-15%20at%2000.22.52.png)

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

model_v2 = YOLO("/kaggle/input/models/natair/chicken-label-drone-model-tolo8m-825frames/pytorch/default/1/best_l.pt")
model_v3 = YOLO("/kaggle/working/runs/chicken_detector_v3_boosted/weights/best.pt")

results_per_img = []
for fp in val_imgs:
    res_v3 = model_v3(str(fp), imgsz=1280, conf=0.25, iou=0.35, verbose=False)
    results_per_img.append((fp, len(res_v3[0].boxes)))

results_per_img.sort(key=lambda x: -x[1])
top_cluster_frames = [fp for fp, n in results_per_img[:6]]

print("Shots with the largest number of chickens (potential candidates for the groups):")
for fp, n in results_per_img[:6]:
    print(f"  {fp.name}: {n} chickens (v3)")

# Comparison of v2 vs v3 in these frames
fig, axes = plt.subplots(2, len(top_cluster_frames), figsize=(6*len(top_cluster_frames), 10))

for col, fp in enumerate(top_cluster_frames):
    img_orig = cv2.imread(str(fp))

    res_v2 = model_v2(str(fp), imgsz=1280, conf=0.25, iou=0.35, verbose=False)
    img_v2 = cv2.cvtColor(img_orig.copy(), cv2.COLOR_BGR2RGB)
    for box in res_v2[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img_v2, (x1, y1), (x2, y2), (255, 0, 0), 2)
    axes[0, col].imshow(img_v2)
    axes[0, col].set_title(f"v2: {len(res_v2[0].boxes)} chickens", fontsize=10)
    axes[0, col].axis("off")

    res_v3 = model_v3(str(fp), imgsz=1280, conf=0.25, iou=0.35, verbose=False)
    img_v3 = cv2.cvtColor(img_orig.copy(), cv2.COLOR_BGR2RGB)
    for box in res_v3[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img_v3, (x1, y1), (x2, y2), (0, 255, 0), 2)
    axes[1, col].imshow(img_v3)
    axes[1, col].set_title(f"v3: {len(res_v3[0].boxes)} chickens", fontsize=10)
    axes[1, col].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/v2_vs_v3_clusters.jpg", dpi=100, bbox_inches="tight")
plt.show()

Shots with the largest number of chickens (potential candidates for the groups):

frame_0025.jpg: 69 chicken (v3)
frame_0030.jpg: 57 chicken (v3)
frame_0034.jpg: 51 chicken (v3)
frame_0029.jpg: 50 chicken (v3)
frame_0010.jpg: 47 chicken (v3)
frame_0009.jpg: 47 chicken (v3)

![](../../data/images/Screenshot%202026-09-15%20at%2000.27.07.png)

In [ ]:
import os

old_dir = "/kaggle/input/datasets/natair/chicken-dataset-v2/chicken_dataset_v2/labels/train"
new_dir = "/kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3/labels/train"

changed = 0
checked = 0
for fname in os.listdir(new_dir):
    old_path = os.path.join(old_dir, fname)
    new_path = os.path.join(new_dir, fname)
    if os.path.isfile(old_path) and os.path.isfile(new_path):
        checked += 1
        old_n = sum(1 for _ in open(old_path))
        new_n = sum(1 for _ in open(new_path))
        if new_n > old_n:
            changed += 1

print(f"Files checked: {checked}")
print(f"Images with added boxes: {changed}")

Files checked: 719
Images with added boxes: 397

In [ ]:
# Duplicating Frames with Additional Markup of Clusters
import os
import shutil
from pathlib import Path

OLD_LABELS_DIR = "/kaggle/input/datasets/natair/chicken-dataset-v2/chicken_dataset_v2/labels/train"
NEW_LABELS_DIR = "/kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3/labels/train"
NEW_IMAGES_DIR = "/kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3/images/train"

OUT_IMAGES_DIR = "/kaggle/working/chicken_dataset_v3_boosted/images/train"
OUT_LABELS_DIR = "/kaggle/working/chicken_dataset_v3_boosted/labels/train"

N_DUPLICATES = 3

os.makedirs(OUT_IMAGES_DIR, exist_ok=True)
os.makedirs(OUT_LABELS_DIR, exist_ok=True)

changed_files = []
checked = 0

for fname in os.listdir(NEW_LABELS_DIR):
    old_path = os.path.join(OLD_LABELS_DIR, fname)
    new_path = os.path.join(NEW_LABELS_DIR, fname)
    if os.path.isfile(old_path) and os.path.isfile(new_path):
        checked += 1
        old_n = sum(1 for _ in open(old_path))
        new_n = sum(1 for _ in open(new_path))
        if new_n > old_n:
            changed_files.append(fname)

print(f"Checked: {checked}  |  Changed (clusters have been relabeled): {len(changed_files)}")

copied_base = 0
for fname in os.listdir(NEW_LABELS_DIR):
    label_src = os.path.join(NEW_LABELS_DIR, fname)
    img_name = Path(fname).stem + ".jpg"
    img_src = os.path.join(NEW_IMAGES_DIR, img_name)

    if not os.path.isfile(img_src):
        print(f"No image found for {fname}")
        continue

    shutil.copy(label_src, os.path.join(OUT_LABELS_DIR, fname))
    shutil.copy(img_src, os.path.join(OUT_IMAGES_DIR, img_name))
    copied_base += 1

print(f"The base train has been copied: {copied_base} pairs")

duplicated = 0
for fname in changed_files:
    stem = Path(fname).stem
    label_src = os.path.join(NEW_LABELS_DIR, fname)
    img_src = os.path.join(NEW_IMAGES_DIR, stem + ".jpg")

    if not os.path.isfile(img_src):
        continue

    for dup_idx in range(1, N_DUPLICATES + 1):
        new_stem = f"{stem}_dup{dup_idx}"
        shutil.copy(label_src, os.path.join(OUT_LABELS_DIR, new_stem + ".txt"))
        shutil.copy(img_src, os.path.join(OUT_IMAGES_DIR, new_stem + ".jpg"))
        duplicated += 1

print(f" Duplicates have been added: {duplicated}")
print(f" Final train: {copied_base + duplicated} files")
print(f" (was {copied_base}, clustered frames are now found in {N_DUPLICATES + 1}x)")

Checked: 719  |  Changed (clusters have been relabeled): 397

The base train has been copied: 719 pairs

Duplicates have been added: 1191

Final train: 1910 files

was 719 , clustered frames are now found in 4x

In [ ]:
import shutil

archive_path = shutil.make_archive(
    "/kaggle/working/chicken_dataset_v3_boosted",
    "zip",
    "/kaggle/working/chicken_dataset_v3_boosted"
)

import os
size_mb = os.path.getsize(archive_path) / 1e6
print(f" The archive was created: {archive_path}")
print(f" Size: {size_mb:.1f} MB")

The archive was created: /kaggle/working/chicken_dataset_v3_boosted.zip
Size: 280.8 MB

In [ ]:
VAL_LABELS_DIR = "/kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3/labels/val"
VAL_IMAGES_DIR = "/kaggle/input/datasets/natair/chicken-dataset-v3-845/chicken_dataset_v3/images/val"

OUT_VAL_IMAGES_DIR = "/kaggle/working/chicken_dataset_v3_boosted/images/val"
OUT_VAL_LABELS_DIR = "/kaggle/working/chicken_dataset_v3_boosted/labels/val"

os.makedirs(OUT_VAL_IMAGES_DIR, exist_ok=True)
os.makedirs(OUT_VAL_LABELS_DIR, exist_ok=True)

for fname in os.listdir(VAL_LABELS_DIR):
    stem = Path(fname).stem
    shutil.copy(os.path.join(VAL_LABELS_DIR, fname), os.path.join(OUT_VAL_LABELS_DIR, fname))
    shutil.copy(os.path.join(VAL_IMAGES_DIR, stem + ".jpg"), os.path.join(OUT_VAL_IMAGES_DIR, stem + ".jpg"))

print(f" val was copied as-is: {len(os.listdir(VAL_LABELS_DIR))} files")

data_yaml_content = f"""
path: /kaggle/working/chicken_dataset_v3_boosted
train: images/train
val: images/val

names:
  0: chicken
"""
with open("/kaggle/working/chicken_dataset_v3_boosted/data.yaml", "w") as f:
    f.write(data_yaml_content)
print(" data.yaml created within the dataset")

val was copied as-is: 126 files

data.yaml created within the dataset